In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpu  = torch.cuda.device_count()
print(f"Device : {device} | GPUs : {n_gpu}")
for i in range(n_gpu):
    print(f"   GPU {i} : {torch.cuda.get_device_name(i)}")

INPUT_DIR  = Path('/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed')
IMAGE_DIR  = INPUT_DIR / 'images' / 'images' / 'image_train'
WORK_DIR   = Path('/kaggle/working')

random_state = 42
torch.manual_seed(random_state)
np.random.seed(random_state)

print(f"\n✅ Image dir existe : {IMAGE_DIR.exists()}")
print(f"   Nb images       : {len(list(IMAGE_DIR.glob('*.jpg'))):,}")

In [ ]:
df_train = pd.read_csv(INPUT_DIR / 'df_train.csv', index_col=0)
df_val   = pd.read_csv(INPUT_DIR / 'df_val.csv',   index_col=0)
df_test  = pd.read_csv(INPUT_DIR / 'df_test.csv',  index_col=0)

# Encodage labels
classes   = sorted(df_train['prdtypecode'].unique())
class2idx = {c: i for i, c in enumerate(classes)}
idx2class = {i: c for i, c in enumerate(classes)}
NUM_CLASSES = len(classes)

df_train['label'] = df_train['prdtypecode'].map(class2idx)
df_val['label']   = df_val['prdtypecode'].map(class2idx)
df_test['label']  = df_test['prdtypecode'].map(class2idx)

# Chemin image
def get_img_path(row):
    return IMAGE_DIR / f"image_{row['imageid']}_product_{row['productid']}.jpg"

# Vérification sur quelques images
ok, missing = 0, 0
for _, row in df_train.head(100).iterrows():
    if get_img_path(row).exists(): ok += 1
    else: missing += 1

print(f"✅ Train : {df_train.shape} | Val : {df_val.shape} | Test : {df_test.shape}")
print(f"   Vérification images (100 premières) : {ok} OK, {missing} manquantes")
print(f"   Classes : {NUM_CLASSES}")

In [ ]:
# Transformations
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

class RakutenImageDataset(Dataset):
    def __init__(self, df, image_dir, transform):
        self.df        = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(row['label'])
        fname = f"image_{row['imageid']}_product_{row['productid']}.jpg"
        fpath = self.image_dir / fname

        try:
            img = Image.open(fpath).convert('RGB')
        except:
            # Image manquante → image blanche
            img = Image.new('RGB', (224, 224), color=(255, 255, 255))

        return self.transform(img), torch.tensor(label, dtype=torch.long)

# DataLoaders
BATCH_SIZE = 64

train_loader = DataLoader(
    RakutenImageDataset(df_train, IMAGE_DIR, train_transforms),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

val_loader = DataLoader(
    RakutenImageDataset(df_val, IMAGE_DIR, val_transforms),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

test_loader = DataLoader(
    RakutenImageDataset(df_test, IMAGE_DIR, val_transforms),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"✅ DataLoaders créés")
print(f"   Train : {len(train_loader)} batches")
print(f"   Val   : {len(val_loader)} batches")

In [ ]:
def build_vgg16(num_classes):
    model = models.vgg16(weights='IMAGENET1K_V1')

    # Freeze toutes les couches convolutives
    for param in model.features.parameters():
        param.requires_grad = False

    # Dégel des 3 derniers blocs conv pour fine-tuning
    for layer in list(model.features.children())[-8:]:
        for param in layer.parameters():
            param.requires_grad = True

    # Remplacement du classifier
    model.classifier = nn.Sequential(
        nn.Linear(25088, 4096),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(4096, 1024),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(1024, num_classes)
    )
    return model

model = build_vgg16(NUM_CLASSES)

if n_gpu > 1:
    model = nn.DataParallel(model)
    print(f"✅ DataParallel sur {n_gpu} GPUs")

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ VGG16 chargé")
print(f"   Params entraînés : {trainable:,} / {total:,}")

In [ ]:
EPOCHS = 10
LR     = 1e-4

# Learning rates différenciés : plus faible pour les couches pré-entraînées
features_params   = [p for n, p in model.named_parameters()
                     if 'features' in n and p.requires_grad]
classifier_params = [p for n, p in model.named_parameters()
                     if 'classifier' in n and p.requires_grad]

optimizer = AdamW([
    {'params': features_params,   'lr': LR * 0.1},  # 1e-5 pour conv
    {'params': classifier_params, 'lr': LR},          # 1e-4 pour FC
], weight_decay=1e-4)

scheduler  = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
criterion  = nn.CrossEntropyLoss()

print(f"✅ Optimiseur configuré")
print(f"   Epochs  : {EPOCHS}")
print(f"   LR FC   : {LR}")
print(f"   LR Conv : {LR*0.1}")
print(f"\n⏱️  Temps estimé : ~15 min/epoch → ~2h30 total")

In [ ]:
def evaluate(model, loader, device, criterion):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss    = criterion(outputs, labels)
            total_loss += loss.item()

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    f1      = f1_score(all_labels, all_preds, average='weighted')
    avg_loss = total_loss / len(loader)
    return f1, avg_loss, np.array(all_probs), np.array(all_preds)

In [ ]:
history     = []
best_f1_val = 0
best_epoch  = 0

print("🚀 Début entraînement VGG16...\n")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        loop.set_postfix(loss=f"{loss.item():.4f}")

    scheduler.step()

    train_f1 = f1_score(all_labels, all_preds, average='weighted')
    avg_loss = total_loss / len(train_loader)
    val_f1, val_loss, _, _ = evaluate(model, val_loader, device, criterion)

    history.append({'epoch': epoch+1, 'train_loss': avg_loss,
                    'train_f1': train_f1, 'val_f1': val_f1})

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"   Train Loss : {avg_loss:.4f}  |  Train F1 : {train_f1:.4f}")
    print(f"   Val   F1   : {val_f1:.4f}")

    if val_f1 > best_f1_val:
        best_f1_val = val_f1
        best_epoch  = epoch + 1
        m = model.module if hasattr(model, 'module') else model
        torch.save(m.state_dict(), WORK_DIR / 'vgg16_best.pth')
        print(f"   💾 Meilleur modèle sauvegardé (F1={val_f1:.4f})")

print(f"\n✅ Entraînement terminé")
print(f"   Meilleur epoch : {best_epoch} | F1 Val : {best_f1_val:.4f}")

In [ ]:
# Rechargement meilleur modèle
best_model = build_vgg16(NUM_CLASSES)
state_dict = torch.load(WORK_DIR / 'vgg16_best.pth', map_location=device)
best_model.load_state_dict(state_dict)
if n_gpu > 1:
    best_model = nn.DataParallel(best_model)
best_model = best_model.to(device)

# Évaluation
val_f1,  _, val_probs,  val_preds  = evaluate(best_model, val_loader,  device, criterion)
test_f1, _, test_probs, test_preds = evaluate(best_model, test_loader, device, criterion)

print(f"✅ Résultats finaux VGG16")
print(f"   F1 Val  : {val_f1:.4f}")
print(f"   F1 Test : {test_f1:.4f}")
print(f"\n📊 Comparaison modèles texte vs image :")
print(f"   TF-IDF + SVC  → {0.8121:.4f}")
print(f"   CamemBERT     → {0.8826:.4f}")
print(f"   VGG16         → {test_f1:.4f}")

# Sauvegarde probabilités pour la fusion
np.save(WORK_DIR / 'vgg16_val_probs.npy',  val_probs)
np.save(WORK_DIR / 'vgg16_test_probs.npy', test_probs)
pd.DataFrame(history).to_csv(WORK_DIR / 'vgg16_history.csv', index=False)

print(f"\n✅ Fichiers sauvegardés :")
for f in sorted(WORK_DIR.iterdir()):
    print(f"   {f.name:<45} {f.stat().st_size/1024/1024:.2f} MB")